# 21 - Introduccion a ciencia de datos: visualizacion, densidad y CRISP-DM

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender ciencia de datos como un proceso para responder preguntas con datos, antes de usar algoritmos.
2. Aplicar las fases de CRISP-DM a ejemplos simples y cercanos.
3. Interpretar histogramas para describir una variable numerica.
4. Interpretar scatter plots para observar relaciones entre dos variables.
5. Relacionar histogramas normalizados con la idea de funcion de densidad de probabilidad.
6. Crear regiones de interes en una grafica y explicar por que ayudan a analizar datos.
7. Resolver ejercicios teoricos y practicos que preparan el camino para clases de modelado.

> En esta clase todavia no entrenamos algoritmos. Primero necesitamos aprender a mirar datos con cuidado.

## Ruta de la sesion

1. Ciencia de datos antes de los algoritmos.
2. CRISP-DM como mapa de trabajo.
3. Dataset simple: informacion academica simulada.
4. Histogramas: frecuencia, forma y variabilidad.
5. Histograma normalizado y funcion de densidad de probabilidad.
6. Scatter plots: relacion entre dos variables.
7. Regiones de interes: por que se buscan y como se crean.
8. CRISP-DM completo con un ejemplo sencillo.
9. Ejercicios teoricos y practicos.

## 1) Ciencia de datos antes de los algoritmos

Ciencia de datos no empieza con modelos. Empieza con una pregunta.

Ejemplos:

- Que estudiantes podrian necesitar apoyo academico?
- En que horarios se llena mas una biblioteca?
- Como cambia la asistencia a clase durante el semestre?
- Que variables parecen relacionarse con una calificacion final?

Antes de pensar en algoritmos conviene responder:

1. Que decision queremos mejorar?
2. Que datos tenemos?
3. Que significa cada columna?
4. Hay datos faltantes, errores o sesgos?
5. Que graficas ayudan a entender el problema?
6. Que accion real se tomaria con el analisis?

En esta sesion trabajaremos con visualizacion y resumenes. Esa base hace que las clases posteriores de algoritmos tengan sentido.

## 2) CRISP-DM en lenguaje practico

CRISP-DM significa *Cross-Industry Standard Process for Data Mining*.

No es un algoritmo. Es una forma ordenada de trabajar en proyectos de datos.

Sus 6 fases son:

1. **Entendimiento del negocio o problema**: cual decision o pregunta importa.
2. **Entendimiento de los datos**: que datos existen, que significan y que problemas tienen.
3. **Preparacion de datos**: limpiar, seleccionar columnas, crear variables utiles.
4. **Modelado**: construir una representacion util del problema. En esta clase puede ser una grafica, una tabla o una regla descriptiva simple; no necesariamente un algoritmo.
5. **Evaluacion**: decidir si el resultado responde la pregunta inicial.
6. **Despliegue**: comunicar o usar el resultado en una decision real.

CRISP-DM suele ser iterativo: si una grafica revela que faltan datos, regresamos a fases anteriores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

crisp_dm = pd.DataFrame({
    "fase": [
        "1. Entendimiento del problema",
        "2. Entendimiento de los datos",
        "3. Preparacion de datos",
        "4. Modelado o representacion",
        "5. Evaluacion",
        "6. Despliegue",
    ],
    "pregunta_guia": [
        "Que decision queremos mejorar?",
        "Que columnas tenemos y que significan?",
        "Que limpiamos, filtramos o transformamos?",
        "Que grafica, tabla o regla simple resume el patron?",
        "El resultado ayuda a responder la pregunta?",
        "Como se comunica o usa el hallazgo?",
    ],
    "producto_en_esta_sesion": [
        "Pregunta clara y alcance limitado",
        "Revision de variables, rangos y posibles sesgos",
        "Tabla ordenada y variables derivadas sencillas",
        "Histogramas, scatter plots y regiones de interes",
        "Interpretacion escrita, no solo graficas bonitas",
        "Reporte breve o recomendacion para una accion",
    ],
})

crisp_dm

## 3) Datos de trabajo: encuesta academica simulada

Usaremos un dataset simulado de estudiantes universitarios.

Cada fila representa un estudiante. Las columnas son:

- `horas_estudio`: horas promedio de estudio por semana para una materia.
- `horas_sueno`: horas promedio de sueno por noche.
- `asistencia_pct`: porcentaje aproximado de asistencia.
- `traslado_min`: minutos de traslado a la universidad.
- `calificacion`: calificacion final aproximada.

Es simulado para aprender sin exponer datos personales. Aun asi, lo analizaremos como si fuera un caso real.

In [ ]:
rng = np.random.default_rng(21)
n = 180

horas_estudio = rng.normal(loc=4.2, scale=1.7, size=n).clip(0.2, 9.5)
horas_sueno = rng.normal(loc=6.7, scale=1.0, size=n).clip(3.5, 9.5)
asistencia_pct = rng.normal(loc=83, scale=10, size=n).clip(45, 100)
traslado_min = rng.gamma(shape=3.0, scale=12.0, size=n).clip(5, 120)
ruido = rng.normal(loc=0, scale=8, size=n)

calificacion = (
    38
    + 5.6 * horas_estudio
    + 0.24 * asistencia_pct
    + 1.5 * (horas_sueno - 6.5)
    - 0.04 * traslado_min
    + ruido
).clip(0, 100)

df = pd.DataFrame({
    "estudiante": np.arange(1, n + 1),
    "horas_estudio": horas_estudio.round(1),
    "horas_sueno": horas_sueno.round(1),
    "asistencia_pct": asistencia_pct.round(1),
    "traslado_min": traslado_min.round(0).astype(int),
    "calificacion": calificacion.round(1),
})

df.head()

In [ ]:
resumen = df[[
    "horas_estudio",
    "horas_sueno",
    "asistencia_pct",
    "traslado_min",
    "calificacion",
]].describe().T

resumen

## 4) Histograma: ver una variable

Un histograma muestra como se distribuye una variable numerica.

La idea es dividir el eje en intervalos llamados `bins` y contar cuantos datos caen en cada intervalo.

Preguntas utiles:

- Donde se concentra la mayoria de valores?
- La distribucion es simetrica o esta cargada hacia un lado?
- Hay valores extremos?
- Que tan dispersos estan los datos?

Primero observaremos la distribucion de `calificacion`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))

bins_calificacion = np.arange(0, 105, 10)
ax.hist(
    df["calificacion"],
    bins=bins_calificacion,
    color="tab:blue",
    edgecolor="white",
)

media = df["calificacion"].mean()
mediana = df["calificacion"].median()

ax.axvline(media, color="tab:red", linewidth=2, label=f"media = {media:.1f}")
ax.axvline(mediana, color="tab:green", linewidth=2, linestyle="--", label=f"mediana = {mediana:.1f}")

ax.set_title("Histograma de calificaciones")
ax.set_xlabel("calificacion")
ax.set_ylabel("numero de estudiantes")
ax.legend()
plt.show()

### Como leer este histograma

- Cada barra representa un intervalo, por ejemplo de 70 a 80.
- La altura indica cuantos estudiantes estan dentro de ese intervalo.
- La media y la mediana ayudan a ubicar el centro de los datos.
- Si una barra esta muy aislada, podria indicar un valor extremo o una zona poco frecuente.

Cambiar el ancho de los bins puede cambiar la apariencia del histograma. Por eso no conviene interpretar una sola grafica como verdad absoluta.

## 5) Del histograma a la funcion de densidad de probabilidad

Para variables continuas, como una calificacion o tiempo de traslado, normalmente no preguntamos:

- Cual es la probabilidad exacta de obtener 73.000000?

Preguntamos por rangos:

- Que proporcion esta entre 70 y 80?
- Que proporcion esta por debajo de 60?
- Que proporcion esta arriba de 90?

Una funcion de densidad de probabilidad describe como se reparte la probabilidad sobre una linea numerica.

La idea clave es:

$$P(a \leq X \leq b) \approx \text{area bajo la densidad entre } a \text{ y } b$$

Un histograma normalizado aproxima esa densidad. La altura de una barra no es la probabilidad completa; la probabilidad aproximada es:

$$\text{area de la barra} = \text{altura} \times \text{ancho del intervalo}$$

In [ ]:
densidades, bordes = np.histogram(
    df["calificacion"],
    bins=bins_calificacion,
    density=True,
)
anchos = np.diff(bordes)
areas = densidades * anchos

tabla_densidad = pd.DataFrame({
    "intervalo": [f"{int(a)} a {int(b)}" for a, b in zip(bordes[:-1], bordes[1:])],
    "densidad_aprox": densidades.round(4),
    "ancho": anchos.astype(int),
    "area_aprox": areas.round(4),
})

print("Suma de areas aproximadas:", areas.sum().round(4))
tabla_densidad

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))

ax.hist(
    df["calificacion"],
    bins=bins_calificacion,
    density=True,
    color="tab:blue",
    alpha=0.65,
    edgecolor="white",
    label="histograma normalizado",
)

ax.axvspan(70, 90, color="tab:orange", alpha=0.25, label="region: 70 a 90")

proporcion_70_90 = ((df["calificacion"] >= 70) & (df["calificacion"] < 90)).mean()

ax.set_title(f"Densidad aproximada: P(70 <= calificacion < 90) = {proporcion_70_90:.2f}")
ax.set_xlabel("calificacion")
ax.set_ylabel("densidad aproximada")
ax.legend()
plt.show()

Lectura importante:

1. Un histograma de frecuencias responde "cuantos casos hay".
2. Un histograma normalizado responde "como se reparte la probabilidad aproximada".
3. En densidad, el area de una region es lo que se interpreta como probabilidad.
4. Mientras mas datos y mejor eleccion de intervalos tengamos, mejor puede aproximarse la forma general de la distribucion.

Esta intuicion sera necesaria cuando despues hablemos de modelos, incertidumbre y regiones de decision.

## 6) Scatter plot: relacion entre dos variables

Un scatter plot muestra puntos en un plano.

Cada punto representa una observacion. En este caso, cada punto es un estudiante.

Usaremos:

- eje x: `horas_estudio`
- eje y: `calificacion`

Preguntas utiles:

- Cuando una variable sube, la otra tiende a subir o bajar?
- La relacion parece fuerte o debil?
- Hay estudiantes que se salen del patron general?
- Hay zonas donde se acumulan muchos puntos?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    df["horas_estudio"],
    df["calificacion"],
    c=df["asistencia_pct"],
    cmap="viridis",
    alpha=0.78,
    edgecolor="white",
    linewidth=0.4,
)

ax.set_title("Scatter plot: estudio, asistencia y calificacion")
ax.set_xlabel("horas de estudio por semana")
ax.set_ylabel("calificacion")

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("asistencia (%)")

plt.show()

### Como leer este scatter plot

- Si los puntos suben de izquierda a derecha, hay relacion positiva.
- Si bajan de izquierda a derecha, hay relacion negativa.
- Si no hay forma clara, la relacion puede ser debil o depender de otras variables.
- El color agrega una tercera variable sin crear otra grafica.
- Un punto raro puede ser un error, un caso especial o una pista importante.

Cuidado: una relacion visual no prueba causalidad. Que dos variables se muevan juntas no significa automaticamente que una cause la otra.

In [ ]:
correlaciones = df[[
    "horas_estudio",
    "horas_sueno",
    "asistencia_pct",
    "traslado_min",
    "calificacion",
]].corr(numeric_only=True)

correlaciones[["calificacion"]].sort_values("calificacion", ascending=False)

La correlacion resume la direccion y fuerza de una relacion lineal entre dos variables.

- Cerca de 1: tienden a subir juntas.
- Cerca de -1: una tiende a subir cuando la otra baja.
- Cerca de 0: no hay relacion lineal clara.

No reemplaza a la grafica. La grafica muestra forma, dispersion y casos raros que un solo numero puede esconder.

## 7) Regiones de interes

Muchas veces buscamos regiones en una grafica porque una nube de puntos completa puede ser dificil de interpretar.

Una region de interes es una zona del grafico que tiene significado para la pregunta.

Ejemplos:

- calificaciones menores a 60: posible riesgo academico.
- muchas horas de estudio pero baja calificacion: revisar habitos, evaluacion o contexto.
- poca asistencia y baja calificacion: posible seguimiento temprano.
- alta asistencia y alta calificacion: patron de buen desempeno.

Estas regiones no son algoritmos de prediccion. Son una forma humana y transparente de ordenar el analisis.

In [ ]:
condiciones = [
    df["calificacion"] < 60,
    (df["horas_estudio"] >= 5) & (df["calificacion"] < 70),
    (df["horas_estudio"] < 3) & (df["calificacion"] >= 75),
    (df["horas_estudio"] >= 5) & (df["calificacion"] >= 80),
]

nombres_region = [
    "riesgo academico",
    "mucho estudio bajo resultado",
    "buen resultado con poco estudio",
    "zona fuerte",
]

df["region_interes"] = np.select(condiciones, nombres_region, default="zona intermedia")

resumen_regiones = (
    df.groupby("region_interes")
    .agg(
        estudiantes=("estudiante", "count"),
        calificacion_media=("calificacion", "mean"),
        estudio_medio=("horas_estudio", "mean"),
        asistencia_media=("asistencia_pct", "mean"),
    )
    .round(1)
    .sort_values("estudiantes", ascending=False)
)

resumen_regiones

In [ ]:
colores_region = {
    "riesgo academico": "tab:red",
    "mucho estudio bajo resultado": "tab:orange",
    "buen resultado con poco estudio": "tab:purple",
    "zona fuerte": "tab:green",
    "zona intermedia": "tab:gray",
}

fig, ax = plt.subplots(figsize=(8.5, 5.5))

for region, datos_region in df.groupby("region_interes"):
    ax.scatter(
        datos_region["horas_estudio"],
        datos_region["calificacion"],
        label=region,
        color=colores_region[region],
        alpha=0.78,
        edgecolor="white",
        linewidth=0.4,
    )

ax.axhspan(0, 60, color="tab:red", alpha=0.08)
ax.axvspan(5, df["horas_estudio"].max() + 0.5, color="tab:green", alpha=0.05)
ax.axvline(5, color="black", linewidth=1, linestyle="--", alpha=0.5)
ax.axhline(60, color="black", linewidth=1, linestyle="--", alpha=0.5)
ax.axhline(80, color="black", linewidth=1, linestyle=":", alpha=0.5)

ax.set_title("Regiones de interes en el scatter plot")
ax.set_xlabel("horas de estudio por semana")
ax.set_ylabel("calificacion")
ax.legend(loc="lower right", fontsize=8)
plt.show()

### Por que crear regiones

Crear regiones ayuda a:

1. Convertir una grafica en preguntas concretas.
2. Comparar subgrupos sin perder de vista el contexto.
3. Detectar casos que merecen revision.
4. Comunicar hallazgos a personas que no programan.
5. Preparar la intuicion para clases posteriores, donde algunos algoritmos construyen regiones de forma automatica.

La clave es justificar cada region: no basta con dibujar lineas; hay que explicar que significan y que decision apoyan.

## 8) CRISP-DM completo con el ejemplo academico

Pregunta guia:

> Como podemos identificar patrones sencillos asociados con estudiantes que podrian necesitar apoyo academico?

En esta sesion la respuesta no sera un algoritmo. Sera un analisis exploratorio con graficas, resumenes y regiones de interes.

In [ ]:
crisp_ejemplo = pd.DataFrame({
    "fase": [
        "Entendimiento del problema",
        "Entendimiento de los datos",
        "Preparacion de datos",
        "Modelado o representacion",
        "Evaluacion",
        "Despliegue",
    ],
    "aplicacion_en_el_ejemplo": [
        "Detectar patrones que ayuden a orientar apoyo academico temprano.",
        "Revisar variables: estudio, sueno, asistencia, traslado y calificacion.",
        "Crear una tabla limpia y revisar rangos; derivar region_interes.",
        "Usar histogramas, scatter plots y regiones transparentes.",
        "Ver si las regiones son comprensibles y utiles para decidir seguimiento.",
        "Preparar un reporte breve para docentes o tutores, sin automatizar decisiones sensibles.",
    ],
    "pregunta_de_control": [
        "Que accion concreta podria tomar la universidad?",
        "Falta alguna variable importante o hay sesgos?",
        "Las variables estan en unidades claras?",
        "La representacion se entiende sin explicar codigo?",
        "El analisis evita conclusiones causales exageradas?",
        "Quien recibe el resultado y cada cuanto se actualiza?",
    ],
})

crisp_ejemplo

### Otro ejemplo simple: biblioteca universitaria

Supongamos que la biblioteca quiere saber cuando abrir mas ventanillas de prestamo.

CRISP-DM podria verse asi:

1. **Problema**: reducir tiempos de espera.
2. **Datos**: hora, dia, numero de prestamos, usuarios en fila, tiempo de atencion.
3. **Preparacion**: quitar registros duplicados, agrupar por hora, revisar dias sin captura.
4. **Representacion**: histogramas de tiempos de espera y scatter plot entre usuarios en fila y espera.
5. **Evaluacion**: verificar si las horas detectadas coinciden con la experiencia del personal.
6. **Despliegue**: recomendar horarios con mas personal durante semanas pico.

Este flujo ya es ciencia de datos aunque no haya un algoritmo de aprendizaje automatico.

## 9) Ejercicios teoricos

Responde con texto breve y ejemplos propios.

### Ejercicio teorico 1: CRISP-DM

Elige uno de estos casos:

- asistencia a clases,
- demanda de comedor universitario,
- prestamos de biblioteca,
- tiempos de traslado al campus.

Para cada fase de CRISP-DM escribe:

1. una pregunta guia,
2. una variable que necesitarias,
3. un riesgo o limitacion.

### Ejercicio teorico 2: histograma

Explica con tus palabras:

1. que representa un bin,
2. por que cambiar el ancho del bin puede cambiar la interpretacion,
3. que diferencia hay entre frecuencia y densidad.

### Ejercicio teorico 3: funcion de densidad

Contesta:

1. Por que en una variable continua hablamos de rangos y no de valores exactos?
2. Que representa el area bajo una densidad entre dos valores?
3. Por que la altura de una barra normalizada no debe leerse sola como probabilidad?

### Ejercicio teorico 4: scatter plot

Observa el scatter plot de estudio y calificacion. Describe:

1. direccion general de la relacion,
2. dispersion de los puntos,
3. un posible caso raro,
4. una variable adicional que podria ayudar a explicar mejor el fenomeno.

### Ejercicio teorico 5: regiones

Propone dos regiones de interes para un problema real. Para cada region explica:

1. donde estaria en la grafica,
2. que significa,
3. que accion podria motivar.

## 10) Ejercicios practicos

Usa las siguientes celdas como punto de partida. Antes de ejecutar, escribe en una celda Markdown que esperas observar.

### Practica 1: cambiar bins del histograma

Cambia `ancho_bin` a 5, 10, 15 y 20.

Responde:

1. Con que ancho se entiende mejor la distribucion?
2. Que detalles aparecen o desaparecen?
3. Cual elegirias para explicar el resultado a alguien no tecnico?

In [ ]:
ancho_bin = 10
bins = np.arange(0, 100 + ancho_bin, ancho_bin)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(df["calificacion"], bins=bins, color="tab:blue", edgecolor="white")
ax.set_title(f"Histograma con bins de ancho {ancho_bin}")
ax.set_xlabel("calificacion")
ax.set_ylabel("numero de estudiantes")
plt.show()

### Practica 2: calcular probabilidad aproximada por rango

Cambia `limite_inferior` y `limite_superior`.

Interpreta el resultado como proporcion de estudiantes dentro del rango.

In [ ]:
limite_inferior = 70
limite_superior = 80

mascara = (
    (df["calificacion"] >= limite_inferior)
    & (df["calificacion"] < limite_superior)
)

proporcion = mascara.mean()
conteo = mascara.sum()

print(f"Estudiantes en el rango: {conteo} de {len(df)}")
print(f"Proporcion aproximada: {proporcion:.3f}")

### Practica 3: crear otro scatter plot

Cambia las variables del eje x y del eje y.

Opciones sugeridas:

- `horas_sueno` vs `calificacion`
- `asistencia_pct` vs `calificacion`
- `traslado_min` vs `calificacion`

Describe si ves relacion positiva, negativa o debil.

In [ ]:
x = "asistencia_pct"
y = "calificacion"

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df[x], df[y], alpha=0.75, color="tab:cyan", edgecolor="white", linewidth=0.4)
ax.set_title(f"Scatter plot: {x} vs {y}")
ax.set_xlabel(x)
ax.set_ylabel(y)
plt.show()

### Practica 4: definir una region propia

Crea una nueva columna booleana para una region que te parezca importante.

Ejemplos:

- baja asistencia y baja calificacion,
- traslado largo y baja asistencia,
- buen desempeno con pocas horas de estudio.

Despues cuenta cuantos estudiantes caen en esa region.

In [ ]:
df["mi_region"] = (df["asistencia_pct"] < 75) & (df["calificacion"] < 65)

conteo_region = df["mi_region"].sum()
proporcion_region = df["mi_region"].mean()

print("Estudiantes en mi region:", conteo_region)
print(f"Proporcion: {proporcion_region:.3f}")

df.loc[df["mi_region"], [
    "estudiante",
    "horas_estudio",
    "asistencia_pct",
    "calificacion",
]].head()

### Practica 5: mini CRISP-DM propio

Llena esta tabla para un caso que elijas. No uses algoritmos; usa solo datos, graficas y regiones de interes.

In [ ]:
mi_crisp_dm = pd.DataFrame({
    "fase": [
        "Entendimiento del problema",
        "Entendimiento de los datos",
        "Preparacion de datos",
        "Modelado o representacion",
        "Evaluacion",
        "Despliegue",
    ],
    "mi_respuesta": [
        "",
        "",
        "",
        "",
        "",
        "",
    ],
})

mi_crisp_dm

## 11) Cierre

Ideas clave:

1. Ciencia de datos es un proceso para convertir datos en decisiones mejor informadas.
2. CRISP-DM ayuda a ordenar ese proceso desde la pregunta hasta el uso del resultado.
3. Un histograma muestra como se distribuye una variable.
4. Un histograma normalizado aproxima una densidad; la probabilidad se interpreta como area en un rango.
5. Un scatter plot ayuda a observar relaciones, dispersion y casos raros.
6. Las regiones de interes convierten graficas en preguntas y acciones concretas.
7. Antes de entrenar algoritmos, hay que saber describir, visualizar e interpretar datos.

En las siguientes clases, los algoritmos apareceran como una forma mas formal de construir reglas o regiones a partir de datos.